# MNIST MLP3 — MuonClip polar quotient Gram spectrum

This notebook trains the repository's canonical `784 → 512 → 512 → 10` MLP3 on MNIST with MuonClip. At initialization and after every epoch it computes the local Jacobian of the polar quotient projection

\[
\Pi(W)=UV^T,\qquad J_W=D\Pi(W),\qquad G_W=J_W^T J_W.
\]

The spectrum is **not normalized or rebinned**. The raw positive eigenvalues of \(G_W\) are computed exactly from the singular values of each weight matrix and passed directly to WeightWatcher's own `WW_powerlaw.pl_fit`. The zero-mode multiplicity is recorded separately.


In [ ]:
from pathlib import Path
import os, sys, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "baseline" / "rg_baselines").is_dir(): ROOT = (p / "baseline").resolve(); break
    if (p / "rg_baselines").is_dir(): ROOT = p.resolve(); break
if ROOT is None: raise RuntimeError("Run from CalculatedContent/rg_optimizers")
sys.path.insert(0, str(ROOT))
from rg_baselines import MLP3, DEFAULT_BASELINE_SEEDS, MNIST_REFERENCE_SUITE_SLUG
from weightwatcher.WW_powerlaw import pl_fit

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
DATA_DIR = Path(os.environ.get("RG_BASELINE_DATA_DIR", ROOT / "data")).expanduser().resolve()
RUN_ROOT = Path(os.environ.get("RG_BASELINE_RUN_ROOT", ROOT / "runs")).expanduser().resolve() / MNIST_REFERENCE_SUITE_SLUG / "muonclip_polar_quotient_gram"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE)
print("output:", RUN_ROOT)


## MuonClip and experiment configuration


In [ ]:
EPOCHS = int(os.environ.get("MUONCLIP_POLAR_EPOCHS", "30"))
BATCH_SIZE = int(os.environ.get("MUONCLIP_POLAR_BATCH", "256"))
SEEDS = tuple(int(x) for x in os.environ.get("MUONCLIP_POLAR_SEEDS", ",".join(map(str, DEFAULT_BASELINE_SEEDS))).split(","))
MATRIX_LR = float(os.environ.get("MUONCLIP_POLAR_LR", "2e-3"))
AUX_LR = float(os.environ.get("MUONCLIP_POLAR_AUX_LR", "2e-3"))
MOMENTUM = float(os.environ.get("MUONCLIP_POLAR_MOMENTUM", "0.95"))
WEIGHT_DECAY = float(os.environ.get("MUONCLIP_POLAR_WEIGHT_DECAY", "1e-2"))
RMS_SCALE = float(os.environ.get("MUONCLIP_POLAR_RMS_SCALE", "0.20"))
GRAD_CLIP = float(os.environ.get("MUONCLIP_POLAR_GRAD_CLIP", "1.0"))
NS_STEPS = int(os.environ.get("MUONCLIP_POLAR_NS_STEPS", "5"))

@torch.no_grad()
def zeropower(update, steps=5, eps=1e-7):
    transpose = update.shape[0] > update.shape[1]
    x = update.T if transpose else update
    x = x.float() / torch.linalg.vector_norm(x.float()).clamp_min(eps)
    a,b,c = 3.4445,-4.7750,2.0315
    for _ in range(steps):
        g = x @ x.T
        x = a*x + (b*g + c*(g@g)) @ x
    return x.T if transpose else x

class MuonClip(torch.optim.Optimizer):
    def __init__(self, params):
        super().__init__(params, dict(lr=MATRIX_LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY))
    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                state = self.state[p]
                buf = state.setdefault('momentum_buffer', torch.zeros_like(p.grad))
                buf.mul_(group['momentum']).add_(p.grad)
                u = zeropower(buf, NS_STEPS).to(p.dtype)
                u.mul_(RMS_SCALE * math.sqrt(max(p.shape)))
                if group['weight_decay']:
                    p.mul_(1.0 - group['lr'] * group['weight_decay'])
                p.add_(u, alpha=-group['lr'])


## Quotient projection Jacobian and Gram spectrum

For a full-rank \(m\times n\) matrix with singular values \(\sigma_1,\ldots,\sigma_r\), \(r=\min(m,n)\), the nonzero eigenvalues of \(G_W=J_W^T J_W\) are

\[
\lambda_{ij}=\frac{4}{(\sigma_i+\sigma_j)^2},\quad i<j,
\]

plus rectangular transverse modes

\[
\lambda_i=\frac{1}{\sigma_i^2}
\]

with multiplicity \(|m-n|\). The remaining modes are zero. WeightWatcher is applied directly to these raw positive eigenvalues.


In [ ]:
def polar_factor(W):
    U,_,Vt = np.linalg.svd(np.asarray(W, dtype=np.float64), full_matrices=False)
    return U @ Vt

def polar_jacobian_action(W, E):
    W = np.asarray(W, dtype=np.float64); E = np.asarray(E, dtype=np.float64)
    U,s,Vt = np.linalg.svd(W, full_matrices=False); V = Vt.T
    A = U.T @ E @ V
    K = (A - A.T) / (s[:,None] + s[None,:]); np.fill_diagonal(K, 0.0)
    out = U @ K @ V.T
    m,n = W.shape
    if m > n: out += (E - U@(U.T@E)) @ V @ np.diag(1/s) @ V.T
    elif n > m: out += U @ np.diag(1/s) @ U.T @ E @ (np.eye(n) - V@V.T)
    return out

def polar_gram_spectrum(W):
    W = np.asarray(W, dtype=np.float64); m,n = W.shape
    s = np.linalg.svd(W, compute_uv=False); r=min(m,n)
    rot = np.array([4.0/(s[i]+s[j])**2 for i in range(r) for j in range(i+1,r)], dtype=np.float64)
    trans = np.repeat(1.0/(s*s), abs(m-n)) if m != n else np.empty(0)
    positive = np.sort(np.concatenate([rot, trans]))
    return positive, int(m*n - positive.size)

def fit_with_weightwatcher(evals):
    fit = pl_fit(data=np.asarray(evals, dtype=np.float64), xmin=None, xmax=None, verbose=False)
    return dict(alpha=float(fit.alpha), D=float(fit.D), xmin=float(fit.xmin), xmax=float(np.max(evals)), tail_evals=int(np.count_nonzero(evals >= fit.xmin)))

# Small numerical check of D Pi(W)
rng=np.random.default_rng(123)
for shape in [(3,3),(4,3),(3,5)]:
    W=rng.normal(size=shape); E=rng.normal(size=shape); eps=1e-6
    fd=(polar_factor(W+eps*E)-polar_factor(W-eps*E))/(2*eps)
    an=polar_jacobian_action(W,E)
    err=np.linalg.norm(fd-an)/np.linalg.norm(fd)
    print(shape, 'Frechet relative error', err)
    assert err < 1e-7


## MNIST loaders


In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
full_train = datasets.MNIST(str(DATA_DIR), train=True, download=True, transform=transform)
test_set = datasets.MNIST(str(DATA_DIR), train=False, download=True, transform=transform)
g = torch.Generator().manual_seed(20_260_807)
perm = torch.randperm(len(full_train), generator=g).tolist()
val_idx, train_idx = perm[:5000], perm[5000:]
train_set, val_set = Subset(full_train, train_idx), Subset(full_train, val_idx)

def loaders(seed):
    tg=torch.Generator().manual_seed(seed+101)
    common=dict(batch_size=BATCH_SIZE, num_workers=0 if DEVICE.type=='mps' else 0, pin_memory=DEVICE.type=='cuda')
    return (DataLoader(train_set, shuffle=True, generator=tg, **common), DataLoader(train_set, shuffle=False, **common), DataLoader(val_set, shuffle=False, **common), DataLoader(test_set, shuffle=False, **common))

@torch.inference_mode()
def evaluate(model, loader):
    model.eval(); loss=0.0; correct=0; n=0
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE); z=model(x)
        loss += float(F.cross_entropy(z,y,reduction='sum').cpu()); correct += int((z.argmax(1)==y).sum().cpu()); n += y.numel()
    return loss/n, correct/n


## Train and fit the quotient-Gram spectrum with WeightWatcher


In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def analyze(model, seed, epoch):
    rows=[]; spectra={}
    for name in ('fc1','fc2','fc3'):
        W=getattr(model,name).weight.detach().cpu().numpy()
        evals, zeros = polar_gram_spectrum(W)
        fit=fit_with_weightwatcher(evals)
        rows.append(dict(seed=seed, epoch=epoch, layer=name, zero_modes=zeros, positive_modes=len(evals), **fit))
        spectra[f'seed_{seed}__epoch_{epoch:03d}__{name}']=evals
    return rows, spectra

all_perf=[]; all_fits=[]; all_spectra={}
for seed in SEEDS:
    set_seed(seed); train_loader, train_eval, val_loader, test_loader = loaders(seed)
    model=MLP3().to(DEVICE)
    matrices=[p for p in model.parameters() if p.ndim==2]
    biases=[p for p in model.parameters() if p.ndim!=2]
    muon=MuonClip(matrices)
    aux=torch.optim.AdamW(biases, lr=AUX_LR, weight_decay=WEIGHT_DECAY)

    rows,spec=analyze(model,seed,0); all_fits.extend(rows); all_spectra.update(spec)
    for epoch in range(1,EPOCHS+1):
        model.train()
        for x,y in train_loader:
            x,y=x.to(DEVICE),y.to(DEVICE); muon.zero_grad(set_to_none=True); aux.zero_grad(set_to_none=True)
            loss=F.cross_entropy(model(x),y); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); muon.step(); aux.step()
        tr_loss,tr_acc=evaluate(model,train_eval); va_loss,va_acc=evaluate(model,val_loader); te_loss,te_acc=evaluate(model,test_loader)
        all_perf.append(dict(seed=seed,epoch=epoch,train_loss=tr_loss,validation_loss=va_loss,test_loss=te_loss,train_accuracy=tr_acc,validation_accuracy=va_acc,test_accuracy=te_acc))
        rows,spec=analyze(model,seed,epoch); all_fits.extend(rows); all_spectra.update(spec)
        print(f'seed={seed} epoch={epoch}/{EPOCHS} val_loss={va_loss:.5f}')

performance=pd.DataFrame(all_perf)
gram_fits=pd.DataFrame(all_fits)
performance.to_csv(RUN_ROOT/'performance_by_epoch_and_seed.csv',index=False)
gram_fits.to_csv(RUN_ROOT/'polar_quotient_gram_weightwatcher_by_epoch_layer_and_seed.csv',index=False)
np.savez_compressed(RUN_ROOT/'polar_quotient_gram_spectra.npz', **all_spectra)
display(gram_fits.tail(18))


## WeightWatcher alpha and fit quality


In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
for layer,frame in gram_fits.groupby('layer'):
    m=frame.groupby('epoch',as_index=False).alpha.mean(); ax.plot(m.epoch,m.alpha,label=layer)
ax.set(xlabel='Epoch',ylabel='WeightWatcher alpha',title=r'Raw spectrum of $G_W=J_W^T J_W$'); ax.grid(alpha=.25); ax.legend(frameon=False); plt.show()

fig,ax=plt.subplots(figsize=(9,5))
for layer,frame in gram_fits.groupby('layer'):
    m=frame.groupby('epoch',as_index=False).D.mean(); ax.plot(m.epoch,m.D,label=layer)
ax.set(xlabel='Epoch',ylabel='WeightWatcher KS D',title=r'WeightWatcher fit quality for $G_W$'); ax.grid(alpha=.25); ax.legend(frameon=False); plt.show()


## Artifacts

The principal output is `polar_quotient_gram_weightwatcher_by_epoch_layer_and_seed.csv`. It contains, for every layer and checkpoint, the raw positive-mode count, zero-mode count, and WeightWatcher `alpha`, `D`, `xmin`, `xmax`, and fitted-tail size. `polar_quotient_gram_spectra.npz` contains the exact unnormalized positive eigenvalues used in those fits.
